<a href="https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M04/M04_Lab2_Beer_Game.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 30px 36px; border-radius: 14px; margin-bottom: 20px;">
  <h1 style="color: #fff; margin: 0; font-size: 28px;">🍺 M04 Lab 2 — Beer Game: AI Supply Chain Simulation</h1>
  <p style="color: rgba(255,255,255,0.85); margin: 8px 0 0; font-size: 15px;">DADS 5250: Generative AI in Practice &nbsp;|&nbsp; Prof. Dehghani</p>
  <p style="color: rgba(255,255,255,0.65); margin: 6px 0 0; font-size: 13px;">⭐⭐ Difficulty: Intermediate &nbsp;|&nbsp; ⏱ Time: ~15 min</p>
</div>

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="color: #001a70; margin: 0 0 8px;">🎯 Learning Objectives</h3>
  <ol style="margin: 0; color: #1a1a2e; font-size: 14px;">
    <li>Understand the <b>Beer Game</b> and the <b>bullwhip effect</b> in supply chains</li>
    <li>Set up a supply chain simulation environment with inventory, demand, and costs</li>
    <li>Create <b>AI agents</b> using LangChain for each supply chain role</li>
    <li>Run a 20-round simulation and <b>visualize the bullwhip effect</b></li>
    <li>Compare AI agent decisions vs a simple heuristic strategy</li>
  </ol>
</div>

## 📦 Setup

Before we start, let's install the required packages and set up our API connection.

> **🔑 API Key Setup:** Go to the **key icon** (🔑) in the left sidebar of Colab → click **"Add a secret"** → Name: `OPENAI_API_KEY` → Value: your key → Toggle **"Notebook access"** ON.

In [ ]:
# ============================================================
# 📦 Install Dependencies
# ============================================================
!pip install -q --upgrade openai langchain langchain-openai matplotlib numpy
!pip install -q git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils

In [ ]:
# ============================================================
# 🔑 API Key & Client Setup
# ============================================================
import os
from google.colab import userdata
from dads5250 import setup_openai, pp, show_response, show_info, quiz

client = setup_openai()
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import matplotlib.pyplot as plt
import numpy as np
import json, re

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">1️⃣ The Beer Game & Bullwhip Effect</h2>
</div>

The **Beer Distribution Game** was created at MIT in the 1960s to demonstrate how supply chains amplify demand signals.

### The Setup
A simple supply chain with 4 roles, each placing orders to the next:

```
Customer → 🛒 Retailer → 📦 Wholesaler → 🏭 Distributor → ⚙️ Factory
```

### The Bullwhip Effect
Even with **stable customer demand**, order sizes get **amplified** as they move upstream. A small 10% increase in customer demand can become a 50%+ swing at the factory.

| Cause | Example |
|-------|--------|
| **Demand forecasting** | Each player guesses future demand |
| **Order batching** | Players order in large batches to reduce shipping costs |
| **Price fluctuation** | Players stock up when prices are low |
| **Shortage gaming** | Players over-order when they fear shortages |

**Can AI agents do better than humans?** Let's find out!

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">2️⃣ Supply Chain Environment</h2>
</div>

Let's build the simulation environment. Each player has:
- **Inventory** (units in stock)
- **Backlog** (unfulfilled orders)
- **Incoming shipments** (orders arriving from upstream)
- **Costs**: $0.50/unit/week holding cost, $1.00/unit/week backlog cost

In [ ]:
# ============================================================
# 🏗️ Supply Chain Environment
# ============================================================

class SupplyChainNode:
    """A single node in the beer game supply chain."""

    def __init__(self, name, initial_inventory=12):
        self.name = name
        self.inventory = initial_inventory
        self.backlog = 0
        self.incoming_order = 0        # order received from downstream
        self.incoming_shipment = 0     # shipment received from upstream
        self.order_placed = 0          # order placed to upstream
        self.holding_cost = 0.50       # $/unit/week
        self.backlog_cost = 1.00       # $/unit/week
        # History tracking
        self.history = {
            "inventory": [], "backlog": [], "orders_placed": [],
            "orders_received": [], "cost": []
        }

    def receive_shipment(self, qty):
        """Receive shipment from upstream supplier."""
        self.incoming_shipment = qty
        self.inventory += qty

    def fulfill_order(self, demand):
        """Try to fulfill incoming order from downstream."""
        self.incoming_order = demand
        total_demand = demand + self.backlog
        shipped = min(self.inventory, total_demand)
        self.inventory -= shipped
        self.backlog = total_demand - shipped
        return shipped

    def weekly_cost(self):
        """Calculate weekly cost."""
        cost = (self.inventory * self.holding_cost) + (self.backlog * self.backlog_cost)
        return cost

    def record(self):
        """Record state for history."""
        cost = self.weekly_cost()
        self.history["inventory"].append(self.inventory)
        self.history["backlog"].append(self.backlog)
        self.history["orders_placed"].append(self.order_placed)
        self.history["orders_received"].append(self.incoming_order)
        self.history["cost"].append(cost)

    def status_str(self):
        """Return current status as a string for the AI agent."""
        return (
            f"Inventory: {self.inventory} units | "
            f"Backlog: {self.backlog} units | "
            f"Last order received from downstream: {self.incoming_order} units | "
            f"Last shipment from upstream: {self.incoming_shipment} units"
        )

# Generate customer demand: stable at 4, then jumps to 8 at week 5
NUM_ROUNDS = 20
customer_demand = [4]*4 + [8]*16

print("✅ Supply chain environment ready!")
print(f"📊 Simulation: {NUM_ROUNDS} rounds")
print(f"📈 Customer demand: 4 units (weeks 1-4), then 8 units (weeks 5-20)")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">3️⃣ AI Agents for Each Role</h2>
</div>

Each supply chain role gets its own **LLM agent** that decides how many units to order each week. The agent sees its current state and must decide wisely.

In [ ]:
# ============================================================
# 🤖 AI Agent: Order Decision Maker
# ============================================================

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.3)

agent_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are the {role} in a beer distribution supply chain game.

Your goal: minimize total cost (holding + backlog) over 20 weeks.
- Holding cost: $0.50 per unit per week (excess inventory)
- Backlog cost: $1.00 per unit per week (unfulfilled orders)
- Lead time: 2 weeks for orders to arrive

Strategy tips:
- Don't overreact to short-term demand changes
- Consider the 2-week lead time when ordering
- Balance: too much inventory = high holding cost, too little = high backlog cost

You MUST respond with ONLY a JSON object: {{"order": <integer>, "reasoning": "<brief>"}}
Order must be between 0 and 30."""),
    ("user", """Week {week}/{total_weeks}.
Your current state: {status}
Recent order history (last 3 weeks placed): {recent_orders}

How many units do you order from your upstream supplier?""")
])

agent_chain = agent_prompt | llm | StrOutputParser()

def get_ai_order(role, week, node, total_weeks=20):
    """Get AI agent's order decision."""
    recent = node.history["orders_placed"][-3:] if len(node.history["orders_placed"]) >= 3 else node.history["orders_placed"]
    try:
        raw = agent_chain.invoke({
            "role": role,
            "week": week,
            "total_weeks": total_weeks,
            "status": node.status_str(),
            "recent_orders": str(recent) if recent else "None yet"
        })
        # Parse JSON from response
        match = re.search(r'\{.*\}', raw, re.DOTALL)
        if match:
            data = json.loads(match.group())
            return max(0, min(30, int(data["order"])))
    except Exception as e:
        pass
    return 4  # fallback

print("✅ AI agent chain ready!")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">4️⃣ Run the Simulation</h2>
</div>

Now let's run both strategies side by side:
- **AI Agents**: Each role uses an LLM to decide order quantities
- **Simple Heuristic**: Each role orders exactly what it received from downstream ("order-up-to" policy)

In [ ]:
# ============================================================
# 🎮 Run AI Simulation (20 rounds)
# ============================================================

roles = ["Retailer", "Wholesaler", "Distributor", "Factory"]

# --- AI Strategy ---
ai_nodes = {r: SupplyChainNode(r) for r in roles}
# Pipeline of pending shipments (2-week lead time)
ai_pipeline = {r: [4, 4] for r in roles}  # 2 weeks of shipments in transit

print("🤖 Running AI Agent simulation...\n")

for week in range(1, NUM_ROUNDS + 1):
    # 1. Receive shipments (from pipeline)
    for role in roles:
        shipment = ai_pipeline[role].pop(0)
        ai_nodes[role].receive_shipment(shipment)

    # 2. Fulfill demand (cascade: customer → retailer → ... → factory)
    demand = customer_demand[week - 1]
    for role in roles:
        shipped = ai_nodes[role].fulfill_order(demand)
        demand = ai_nodes[role].order_placed  # next node sees this node's order

    # 3. AI decides order quantities
    for i, role in enumerate(roles):
        order = get_ai_order(role, week, ai_nodes[role])
        ai_nodes[role].order_placed = order
        # Add to upstream pipeline (arrives in 2 weeks)
        if i < len(roles) - 1:
            ai_pipeline[roles[i+1]].append(0)  # placeholder
        ai_pipeline[role].append(order)  # own pipeline fills in 2 weeks

    # 4. Record state
    for role in roles:
        ai_nodes[role].record()

    if week % 5 == 0:
        print(f"   Week {week:2d}: ", end="")
        for role in roles:
            print(f"{role[:3]}(inv={ai_nodes[role].inventory:2d}, ord={ai_nodes[role].order_placed:2d}) ", end="")
        print()

print("\n✅ AI simulation complete!")

In [ ]:
# ============================================================
# 📊 Run Heuristic Simulation (baseline)
# ============================================================

heur_nodes = {r: SupplyChainNode(r) for r in roles}
heur_pipeline = {r: [4, 4] for r in roles}

for week in range(1, NUM_ROUNDS + 1):
    for role in roles:
        shipment = heur_pipeline[role].pop(0)
        heur_nodes[role].receive_shipment(shipment)

    demand = customer_demand[week - 1]
    for role in roles:
        shipped = heur_nodes[role].fulfill_order(demand)
        demand = heur_nodes[role].order_placed

    # Heuristic: order exactly what was received from downstream
    for i, role in enumerate(roles):
        order = heur_nodes[role].incoming_order  # pass-through
        heur_nodes[role].order_placed = order
        if i < len(roles) - 1:
            heur_pipeline[roles[i+1]].append(0)
        heur_pipeline[role].append(order)

    for role in roles:
        heur_nodes[role].record()

print("✅ Heuristic simulation complete!")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">5️⃣ Visualize the Bullwhip Effect</h2>
</div>

The key insight: **order variance increases** as we move upstream. Let's see it.

In [ ]:
# ============================================================
# 📈 Bullwhip Effect Visualization (Professional Dark Theme)
# ============================================================

colors = ["#4fc3f7", "#81c784", "#ffb74d", "#ef5350"]
weeks = list(range(1, NUM_ROUNDS + 1))

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.patch.set_facecolor("#1a1a2e")

# --- Plot 1: AI Orders Over Time ---
ax = axes[0, 0]
ax.set_facecolor("#16213e")
ax.plot(weeks, customer_demand, "--", color="white", alpha=0.5, label="Customer Demand", linewidth=2)
for i, role in enumerate(roles):
    ax.plot(weeks, ai_nodes[role].history["orders_placed"], color=colors[i], label=role, linewidth=2)
ax.set_title("AI Agent Orders", color="white", fontsize=14, fontweight="bold")
ax.set_xlabel("Week", color="white")
ax.set_ylabel("Order Quantity", color="white")
ax.legend(fontsize=9, facecolor="#16213e", edgecolor="white", labelcolor="white")
ax.tick_params(colors="white")
ax.grid(alpha=0.2)

# --- Plot 2: Heuristic Orders Over Time ---
ax = axes[0, 1]
ax.set_facecolor("#16213e")
ax.plot(weeks, customer_demand, "--", color="white", alpha=0.5, label="Customer Demand", linewidth=2)
for i, role in enumerate(roles):
    ax.plot(weeks, heur_nodes[role].history["orders_placed"], color=colors[i], label=role, linewidth=2)
ax.set_title("Heuristic Orders", color="white", fontsize=14, fontweight="bold")
ax.set_xlabel("Week", color="white")
ax.set_ylabel("Order Quantity", color="white")
ax.legend(fontsize=9, facecolor="#16213e", edgecolor="white", labelcolor="white")
ax.tick_params(colors="white")
ax.grid(alpha=0.2)

# --- Plot 3: Order Variance (Bullwhip Metric) ---
ax = axes[1, 0]
ax.set_facecolor("#16213e")
ai_var = [np.var(ai_nodes[r].history["orders_placed"]) for r in roles]
heur_var = [np.var(heur_nodes[r].history["orders_placed"]) for r in roles]
x = np.arange(len(roles))
bar_w = 0.35
ax.bar(x - bar_w/2, ai_var, bar_w, label="AI Agents", color="#4fc3f7", edgecolor="white", linewidth=0.5)
ax.bar(x + bar_w/2, heur_var, bar_w, label="Heuristic", color="#ef5350", edgecolor="white", linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(roles, color="white")
ax.set_title("Order Variance (Bullwhip Metric)", color="white", fontsize=14, fontweight="bold")
ax.set_ylabel("Variance", color="white")
ax.legend(fontsize=9, facecolor="#16213e", edgecolor="white", labelcolor="white")
ax.tick_params(colors="white")
ax.grid(alpha=0.2, axis="y")

# --- Plot 4: Cumulative Cost ---
ax = axes[1, 1]
ax.set_facecolor("#16213e")
for i, role in enumerate(roles):
    ai_cum = np.cumsum(ai_nodes[role].history["cost"])
    heur_cum = np.cumsum(heur_nodes[role].history["cost"])
    ax.plot(weeks, ai_cum, color=colors[i], linewidth=2, label=f"{role} (AI)")
    ax.plot(weeks, heur_cum, color=colors[i], linewidth=2, linestyle="--", alpha=0.5)
ax.set_title("Cumulative Cost (solid=AI, dashed=Heuristic)", color="white", fontsize=14, fontweight="bold")
ax.set_xlabel("Week", color="white")
ax.set_ylabel("Total Cost ($)", color="white")
ax.legend(fontsize=8, facecolor="#16213e", edgecolor="white", labelcolor="white")
ax.tick_params(colors="white")
ax.grid(alpha=0.2)

plt.suptitle("🍺 Beer Game: AI Agents vs Heuristic", color="white", fontsize=18, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Print summary
print("\n" + "="*60)
print("📊 TOTAL COST COMPARISON")
print("="*60)
print(f"{'Role':<14} {'AI Cost':>10} {'Heuristic':>10} {'Savings':>10}")
print("-"*44)
for role in roles:
    ai_total = sum(ai_nodes[role].history["cost"])
    heur_total = sum(heur_nodes[role].history["cost"])
    diff = heur_total - ai_total
    print(f"{role:<14} ${ai_total:>8.2f} ${heur_total:>8.2f}  {'✅' if diff > 0 else '❌'} ${diff:>+.2f}")

**📝 Your Observations:** *(double-click to edit)*

1. Look at the "Order Variance" chart. Where does the bullwhip effect peak? Why does variance increase upstream? _[Your answer]_
2. Compare the AI vs Heuristic orders in weeks 5-8 (right after the demand jump). Which strategy reacted faster? _[Your answer]_
3. Did the AI agents reduce total cost compared to the heuristic? By how much? _[Your answer]_

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0; font-size: 14px;">
  <b>🧠 Key Takeaway:</b> The bullwhip effect is clearly visible — order variance <b>increases upstream</b> (Factory has the highest variance). AI agents may dampen or amplify this depending on their strategy. The key insight: even with AI, the <b>information delay</b> (2-week lead time) is the root cause of the bullwhip effect.
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">📝 Knowledge Check</h2>
</div>

In [ ]:
# ============================================================
# 📝 Quiz — Test Your Understanding
# ============================================================

quiz([
    {
        "q": "What is the bullwhip effect?",
        "options": [
            "When customer demand drops to zero",
            "When order variability amplifies as it moves upstream in the supply chain",
            "When all supply chain nodes order the same quantity",
            "When inventory costs decrease over time"
        ],
        "answer": 1,
        "explanation": "The bullwhip effect describes how small changes in customer demand get amplified as orders propagate upstream through the supply chain. Each node overreacts, creating larger and larger swings."
    },
    {
        "q": "Why does the 2-week lead time worsen the bullwhip effect?",
        "options": [
            "It makes shipping more expensive",
            "It creates information delay — nodes can't see real-time demand, so they guess and overreact",
            "It reduces inventory holding costs",
            "It has no effect on the bullwhip"
        ],
        "answer": 1,
        "explanation": "Lead time creates a delay between placing an order and receiving it. During that gap, nodes make decisions based on outdated information, leading to overordering or underordering — which amplifies the bullwhip effect."
    }
])

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">🔧 Hands-On Exercise</h2>
</div>

### Exercise: Modify an Agent's Strategy

The AI agents above used a general strategy. Modify the **Factory** agent to be more conservative — add a rule to its system prompt and compare results.

In [ ]:
# --- Expected Output ---
show_expected('{"order": 8, "reasoning": "Demand is 10 but conservative cap at 1.5x = 15. Ordering 8 to maintain safety stock."}')

In [ ]:
# ============================================================
# 🔧 Exercise: Custom Factory Strategy
# ============================================================
# Replace each ----- with the correct value

# Create a custom prompt for a conservative factory agent
conservative_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are the Factory in a beer distribution supply chain.

Your CONSERVATIVE strategy:
- Never order more than 1.5x your incoming demand
- -----                                                # YOUR CODE: Add a rule about minimum inventory
- Prefer small, steady orders over large swings

Respond with ONLY: {{"order": <integer>, "reasoning": "<brief>"}}
Order must be between 0 and 30."""),
    ("user", """Week {week}. State: {status}
How many units do you order?""")
])

conservative_chain = conservative_prompt | ----- | StrOutputParser()  # What goes here?

# Test it
test_result = conservative_chain.invoke({
    "week": 5,
    "status": "Inventory: 8 units | Backlog: 3 units | Last order received: 10 units | Last shipment: 4 units"
})
print(f"🏭 Conservative Factory says: {test_result}")

**📝 Your Observations:** *(double-click to edit this cell)*

1. Did the AI agents reduce the bullwhip effect compared to the heuristic? _[Your answer]_

2. Which role had the highest cost? Why? _[Your answer]_

3. How would sharing demand information across all nodes change the outcome? _[Your answer]_

<div style="background: #e8f5e9; border-left: 4px solid #4caf50; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="margin: 0 0 8px;">📖 Self-Study Activity</h3>
  <p style="margin: 0; font-size: 14px;">Try these extensions:</p>
  <ol style="font-size: 14px;">
    <li>Change customer demand to a <b>random pattern</b> — does AI adapt better than heuristic?</li>
    <li>Give each agent <b>access to customer demand</b> (information sharing) — how much does it help?</li>
    <li>Increase lead time to <b>4 weeks</b> — how does the bullwhip change?</li>
  </ol>
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 24px 30px; border-radius: 12px; margin: 24px 0 0; text-align: center;">
  <h2 style="color: white; margin: 0 0 10px;">🎉 Lab 2 Complete!</h2>
  <p style="color: rgba(255,255,255,0.85); margin: 0; font-size: 14px;">You've simulated the Beer Game with AI agents, visualized the bullwhip effect, and compared strategies.</p>
  <div style="background: rgba(255,255,255,0.15); border-radius: 8px; padding: 12px; margin-top: 14px; text-align: left;">
    <p style="color: white; margin: 0 0 6px; font-weight: bold; font-size: 14px;">Key Takeaways:</p>
    <ul style="color: rgba(255,255,255,0.9); margin: 0; font-size: 13px;">
      <li>The <b>bullwhip effect</b> amplifies demand variability upstream</li>
      <li>LLM agents can make supply chain decisions using <b>LangChain chains</b></li>
      <li><b>Information delay</b> (lead time) is the root cause — even AI can't fully overcome it</li>
      <li>Comparing strategies with <b>simulation + visualization</b> is a powerful analysis tool</li>
    </ul>
  </div>
  <p style="color: rgba(255,255,255,0.7); margin: 14px 0 0; font-size: 13px;"><b>Next →</b> M05 Lab 1: Embeddings &amp; Vector Stores</p>
</div>